# Distracted Driver Detection

A computer-vision project that classifies images of drivers into 10 behavior categories
(safe driving, texting, talking on the phone, drinking, reaching behind, etc.) using a
ResNet-50 convolutional neural network built from scratch with Keras/TensorFlow.

**Dataset:** [State Farm Distracted Driver Detection (Kaggle)](https://www.kaggle.com/c/state-farm-distracted-driver-detection)

## Classes

| Label | Description              |
|-------|---------------------------|
| c0    | Safe driving               |
| c1    | Texting - right hand       |
| c2    | Talking on the phone - right |
| c3    | Texting - left hand        |
| c4    | Talking on the phone - left |
| c5    | Operating the radio         |
| c6    | Drinking                   |
| c7    | Reaching behind            |
| c8    | Hair and makeup            |
| c9    | Talking to passenger       |

## Pipeline

1. Load and explore the `driver_imgs_list.csv` metadata (class balance, images per subject)
2. Preprocess: shuffle, encode class labels, convert images to numpy arrays
3. Build a ResNet-50 architecture (identity & convolutional blocks) from scratch
4. Train with **Leave-One-Group-Out cross-validation**, grouping by driver (`subject`), to
   avoid leaking a driver's appearance between train and validation splits
5. Evaluate bias/variance across different epoch counts
6. Train a final model and generate predictions on the holdout/test set


## 1. Imports & Configuration

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import LeaveOneGroupOut
from tensorflow.keras import backend as K
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.layers import (
    Activation,
    Add,
    AveragePooling2D,
    BatchNormalization,
    Conv2D,
    Dense,
    Flatten,
    Input,
    MaxPooling2D,
    ZeroPadding2D,
)
from tensorflow.keras.models import Model, load_model, save_model
from tensorflow.keras.preprocessing import image

# Reproducibility
SEED = 0
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Project paths (adjust to match your local / repo layout) ----
DATA_DIR = "data"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "imgs", "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "imgs", "test")
TRAIN_LABELS_CSV = os.path.join(DATA_DIR, "driver_imgs_list.csv")
TEST_FILENAMES_CSV = os.path.join(DATA_DIR, "test_file_names.csv")

ARRAY_DIR = "arrays"      # cached .npy image arrays
MODEL_DIR = "models"      # saved Keras models
RESULTS_DIR = "results"   # prediction outputs

for d in (ARRAY_DIR, MODEL_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS = 64, 64, 3
NUM_CLASSES = 10
CLASS_NAMES = {
    "c0": "safe driving",
    "c1": "texting - right",
    "c2": "talking on phone - right",
    "c3": "texting - left",
    "c4": "talking on phone - left",
    "c5": "operating the radio",
    "c6": "drinking",
    "c7": "reaching behind",
    "c8": "hair and makeup",
    "c9": "talking to passenger",
}


## 2. Helper Functions

In [ ]:
def plot_class_frequency(class_counts, title="Number of Images per Class"):
    """Bar chart of image counts per class."""
    plt.figure(figsize=(12, 4))
    plt.bar(class_counts.index, class_counts.values)
    plt.xlabel("class")
    plt.ylabel("count")
    plt.title(title)
    plt.show()


def describe_image_data(data):
    """Print summary statistics for a Series of per-group image counts."""
    print(f"Average number of images: {np.mean(data):.2f}")
    print(f"Lowest image count: {data.min()} (at {data.idxmin()})")
    print(f"Highest image count: {data.max()} (at {data.idxmax()})")
    print(data.describe())


def create_img_array(height, width, channel, data, folder, save_labels=True, array_dir=ARRAY_DIR, img_root=DATA_DIR):
    """
    Convert a folder of images referenced in `data` into a numpy array of shape
    (examples, height, width, channel) and persist it as a .npy file (so this expensive
    step doesn't need to be re-run on every notebook execution).

    Arguments:
        height, width, channel -- target image dimensions
        data -- DataFrame with columns: subject, classname, img (classname/subject only
                required when folder == 'train' and save_labels is True)
        folder -- 'train' or 'test', selects the image subdirectory
        save_labels -- also persist the integer class labels (train only)
        array_dir -- output directory for the cached .npy files
        img_root -- root directory containing 'imgs/train' and 'imgs/test'
    """
    num_examples = len(data)
    X = np.zeros((num_examples, height, width, channel))
    Y = np.zeros(num_examples) if (folder == "train" and save_labels) else None

    for m in range(num_examples):
        current_img = data["img"].iloc[m]
        img_path = os.path.join(img_root, "imgs", folder, current_img)
        img = image.load_img(img_path, target_size=(height, width))
        x = image.img_to_array(img)
        x = preprocess_input(x)
        X[m] = x
        if Y is not None:
            Y[m] = data.loc[data["img"] == current_img, "classname"].iloc[0]

    np.save(os.path.join(array_dir, f"X_{folder}_{height}_{width}.npy"), X)
    if Y is not None:
        np.save(os.path.join(array_dir, f"Y_{folder}_{height}_{width}.npy"), Y)


def rescale(X):
    """Rescale a preprocessed (ImageNet-style) array back to roughly [0, 1] for display."""
    return (1 / (2 * np.max(X))) * X + 0.5


def print_image(X_scaled, index, Y=None):
    """Display a single image from an array, optionally printing its label/prediction."""
    plt.imshow(X_scaled[index])
    plt.axis("off")
    plt.show()
    if Y is not None:
        label = np.squeeze(Y[index]) if Y.shape[1] == 1 else np.argmax(Y[index])
        print(f"y = {label}")


def leave_one_group_out_cv(X, Y, group, model_fn, input_shape, classes, init, optimizer, metrics, epochs, batch_size):
    """
    Train/evaluate `model_fn` using Leave-One-Group-Out cross-validation, where each
    group is a driver ('subject'). This prevents the same driver's images from
    appearing in both the training and validation folds, giving a more honest estimate
    of how well the model generalizes to unseen drivers.

    Returns a DataFrame indexed by held-out subject id with train/val loss & accuracy.
    """
    logo = LeaveOneGroupOut()
    n_splits = logo.get_n_splits(X, Y, group)
    cv_scores = np.zeros((n_splits, 4))
    subject_ids = []

    for i, (train_idx, val_idx) in enumerate(logo.split(X, Y, group)):
        model = model_fn(input_shape=input_shape, classes=classes, init=init)
        model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=[metrics])
        model.fit(X[train_idx], Y[train_idx], epochs=epochs, batch_size=batch_size, verbose=0)

        train_scores = model.evaluate(X[train_idx], Y[train_idx], verbose=0)
        val_scores = model.evaluate(X[val_idx], Y[val_idx], verbose=0)

        cv_scores[i] = [train_scores[0], train_scores[1] * 100, val_scores[0], val_scores[1] * 100]
        subject_ids.append(group.iloc[val_idx[0]])

        K.clear_session()

    return pd.DataFrame(
        cv_scores, index=subject_ids,
        columns=["Train_loss", "Train_acc", "Val_loss", "Val_acc"],
    )


## 3. Exploratory Data Analysis

Download the dataset from Kaggle and place it under `data/` so the structure looks like:

```
data/
├── driver_imgs_list.csv
├── test_file_names.csv
└── imgs/
    ├── train/   (subfolders c0-c9 or flat, matching driver_imgs_list.csv)
    └── test/
```


In [ ]:
driver_imgs_df = pd.read_csv(TRAIN_LABELS_CSV)
print(f"Training set: {driver_imgs_df.shape[0]} images")
driver_imgs_df.head()


In [ ]:
class_counts = driver_imgs_df["classname"].value_counts().sort_index()
plot_class_frequency(class_counts)
describe_image_data(class_counts)


It's also useful to check the number of images **per driver (subject)** — an imbalance
here could bias the model toward a particular driver's appearance/background rather than
the behavior itself.

In [ ]:
subject_counts = driver_imgs_df["subject"].value_counts()
plt.figure(figsize=(12, 4))
plt.bar(subject_counts.index, subject_counts.values)
plt.xlabel("subject")
plt.ylabel("count")
plt.title("Number of Images per Subject")
plt.xticks(rotation=90)
plt.show()
describe_image_data(subject_counts)


In [ ]:
print("Missing values per column:")
print(driver_imgs_df.isnull().sum())


## 4. Preprocess Labels

In [ ]:
# The data is provided ordered by class (c0..c9) -- shuffle it
driver_imgs_df = driver_imgs_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Map class strings ('c0'..'c9') to integer labels (0..9)
class_to_int = {f"c{i}": i for i in range(NUM_CLASSES)}
driver_imgs_df["classname"] = driver_imgs_df["classname"].map(class_to_int)

driver_imgs_df.head()


## 5. Convert Images to Arrays

Images are resized to 64x64x3 and cached as `.npy` files so this expensive step only runs once.

In [ ]:
X_PATH = os.path.join(ARRAY_DIR, f"X_train_{IMG_HEIGHT}_{IMG_WIDTH}.npy")
Y_PATH = os.path.join(ARRAY_DIR, f"Y_train_{IMG_HEIGHT}_{IMG_WIDTH}.npy")

if not (os.path.exists(X_PATH) and os.path.exists(Y_PATH)):
    create_img_array(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS, driver_imgs_df, "train")

X = np.load(X_PATH)
Y = np.load(Y_PATH)
print(f"X shape: {X.shape}, Y shape: {Y.shape}")


In [ ]:
# Sanity checks
assert (X == 0).sum() == 0, "Unexpected zero-valued pixels found in X"
plot_class_frequency(pd.Series(Y).value_counts().sort_index(), title="Class Distribution After Array Conversion")


In [ ]:
# Visual sanity check: display a sample image and its label
X_scaled = rescale(X)
sample_idx = 2
print_image(X_scaled, sample_idx, Y=Y.reshape(-1, 1))
print(f"Class meaning: {CLASS_NAMES['c' + str(int(Y[sample_idx]))]}")


## 6. Build the Model: ResNet-50

Residual connections allow training much deeper networks without vanishing gradients. We implement the standard identity and convolutional blocks, then assemble the full 50-layer architecture.

In [ ]:
def identity_block(X, f, filters, stage, block, init):
    """
    Identity block: used when the input and output of the block have the same shape,
    so the shortcut can be added directly to the main path.

    Arguments:
        X -- input tensor, shape (m, n_H_prev, n_W_prev, n_C_prev)
        f -- kernel size for the middle CONV layer of the main path
        filters -- list of 3 ints, number of filters in each CONV layer
        stage, block -- used to generate unique layer names
        init -- kernel initializer

    Returns:
        X -- output tensor of the identity block
    """
    conv_name_base = f"res{stage}{block}_branch"
    bn_name_base = f"bn{stage}{block}_branch"
    F1, F2, F3 = filters
    X_shortcut = X

    X = Conv2D(F1, (1, 1), strides=(1, 1), padding="valid", name=conv_name_base + "2a", kernel_initializer=init)(X)
    X = BatchNormalization(axis=3, name=bn_name_base + "2a")(X)
    X = Activation("relu")(X)

    X = Conv2D(F2, (f, f), strides=(1, 1), padding="same", name=conv_name_base + "2b", kernel_initializer=init)(X)
    X = BatchNormalization(axis=3, name=bn_name_base + "2b")(X)
    X = Activation("relu")(X)

    X = Conv2D(F3, (1, 1), strides=(1, 1), padding="valid", name=conv_name_base + "2c", kernel_initializer=init)(X)
    X = BatchNormalization(axis=3, name=bn_name_base + "2c")(X)

    X = Add()([X, X_shortcut])
    X = Activation("relu")(X)
    return X


def convolutional_block(X, f, filters, stage, block, init, s=2):
    """
    Convolutional block: used when the input and output shapes differ, so the shortcut
    path also passes through a (1x1) convolution to match dimensions before the add.

    Arguments mirror `identity_block`, plus `s`, the stride used to downsample.

    Returns:
        X -- output tensor of the convolutional block
    """
    conv_name_base = f"res{stage}{block}_branch"
    bn_name_base = f"bn{stage}{block}_branch"
    F1, F2, F3 = filters
    X_shortcut = X

    X = Conv2D(F1, (1, 1), strides=(s, s), name=conv_name_base + "2a", kernel_initializer=init)(X)
    X = BatchNormalization(axis=3, name=bn_name_base + "2a")(X)
    X = Activation("relu")(X)

    X = Conv2D(F2, (f, f), strides=(1, 1), padding="same", name=conv_name_base + "2b", kernel_initializer=init)(X)
    X = BatchNormalization(axis=3, name=bn_name_base + "2b")(X)
    X = Activation("relu")(X)

    X = Conv2D(F3, (1, 1), strides=(1, 1), name=conv_name_base + "2c", kernel_initializer=init)(X)
    X = BatchNormalization(axis=3, name=bn_name_base + "2c")(X)

    X_shortcut = Conv2D(F3, (1, 1), strides=(s, s), name=conv_name_base + "1", kernel_initializer=init)(X_shortcut)
    X_shortcut = BatchNormalization(axis=3, name=bn_name_base + "1")(X_shortcut)

    X = Add()([X, X_shortcut])
    X = Activation("relu")(X)
    return X


def ResNet50(input_shape=(64, 64, 3), classes=10, init=None):
    """
    ResNet-50 architecture:
    CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> CONVBLOCK -> IDBLOCK*2 -> CONVBLOCK -> IDBLOCK*3
    -> CONVBLOCK -> IDBLOCK*5 -> CONVBLOCK -> IDBLOCK*2 -> AVGPOOL -> FC

    Arguments:
        input_shape -- shape of input images
        classes -- number of output classes
        init -- kernel initializer (defaults to Glorot uniform)

    Returns:
        model -- a compiled-ready Keras Model instance
    """
    if init is None:
        init = glorot_uniform(seed=SEED)

    X_input = Input(input_shape)
    X = ZeroPadding2D((3, 3))(X_input)

    # Stage 1
    X = Conv2D(64, (7, 7), strides=(2, 2), name="conv1", kernel_initializer=init)(X)
    X = BatchNormalization(axis=3, name="bn_conv1")(X)
    X = Activation("relu")(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    # Stage 2
    X = convolutional_block(X, f=3, filters=[64, 64, 256], stage=2, block="a", s=1, init=init)
    X = identity_block(X, 3, [64, 64, 256], stage=2, block="b", init=init)
    X = identity_block(X, 3, [64, 64, 256], stage=2, block="c", init=init)

    # Stage 3
    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, block="a", s=2, init=init)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block="b", init=init)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block="c", init=init)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block="d", init=init)

    # Stage 4
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, block="a", s=2, init=init)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block="b", init=init)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block="c", init=init)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block="d", init=init)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block="e", init=init)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block="f", init=init)

    # Stage 5
    X = convolutional_block(X, f=3, filters=[512, 512, 2048], stage=5, block="a", s=2, init=init)
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block="b", init=init)
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block="c", init=init)

    # Output head
    X = AveragePooling2D(pool_size=(2, 2), name="avg_pool")(X)
    X = Flatten()(X)
    X = Dense(classes, activation="softmax", name="fc" + str(classes), kernel_initializer=init)(X)

    model = Model(inputs=X_input, outputs=X, name="ResNet50")
    return model


## 7. Cross-Validation Training (Leave-One-Group-Out)

We group by driver (`subject`) so that the validation fold always contains a driver the
model has never seen during training — a much more realistic estimate of real-world
performance than a random split, since random splits would let the model "memorize"
a driver's face/background/clothing rather than learn the underlying behavior.

In [ ]:
# Normalize images and reshape labels
X_train = X / 255.0
Y_train = np.expand_dims(Y.astype(int), -1)

print(f"Number of training examples = {X_train.shape[0]}")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")


Below we sweep the number of epochs (2 -> 5 -> 10) to study the bias/variance trade-off.
**Note:** running LOGO cross-validation trains one ResNet-50 per held-out driver (~26 models)
and is computationally expensive — consider running on a GPU, or reducing to a handful of
drivers/epochs for a quick smoke test.

In [ ]:
scores_2ep = leave_one_group_out_cv(
    X_train, Y_train, group=driver_imgs_df["subject"],
    model_fn=ResNet50, input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), classes=NUM_CLASSES,
    init=glorot_uniform(seed=SEED), optimizer="adam", metrics="accuracy",
    epochs=2, batch_size=32,
)

print("Train acc: {:.2f}. Val acc: {:.2f}".format(scores_2ep["Train_acc"].mean(), scores_2ep["Val_acc"].mean()))
print("Train loss: {:.2f}. Val loss: {:.2f}".format(scores_2ep["Train_loss"].mean(), scores_2ep["Val_loss"].mean()))


In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(scores_2ep.index, scores_2ep["Val_acc"].sort_values(ascending=False))
plt.ylabel("Validation Accuracy (%)")
plt.title("Per-Driver Validation Accuracy (2 epochs)")
plt.xticks(rotation=90)
plt.show()
scores_2ep.describe()


In [ ]:
scores_5ep = leave_one_group_out_cv(
    X_train, Y_train, group=driver_imgs_df["subject"],
    model_fn=ResNet50, input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), classes=NUM_CLASSES,
    init=glorot_uniform(seed=SEED), optimizer="adam", metrics="accuracy",
    epochs=5, batch_size=32,
)
print("Train acc: {:.2f}. Val acc: {:.2f}".format(scores_5ep["Train_acc"].mean(), scores_5ep["Val_acc"].mean()))
print("Train loss: {:.2f}. Val loss: {:.2f}".format(scores_5ep["Train_loss"].mean(), scores_5ep["Val_loss"].mean()))


In [ ]:
scores_10ep = leave_one_group_out_cv(
    X_train, Y_train, group=driver_imgs_df["subject"],
    model_fn=ResNet50, input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), classes=NUM_CLASSES,
    init=glorot_uniform(seed=SEED), optimizer="adam", metrics="accuracy",
    epochs=10, batch_size=32,
)
print("Train acc: {:.2f}. Val acc: {:.2f}".format(scores_10ep["Train_acc"].mean(), scores_10ep["Val_acc"].mean()))
print("Train loss: {:.2f}. Val loss: {:.2f}".format(scores_10ep["Train_loss"].mean(), scores_10ep["Val_loss"].mean()))


### Bias / Variance Summary

| Epochs | Train Acc | Val Acc | Bias (100% - Train Acc) | Variance (Train Acc - Val Acc) |
|--------|-----------|---------|--------------------------|----------------------------------|
| 2      | ~31%      | ~25%    | high                     | low                              |
| 5      | ~38%      | ~26%    | high                     | growing                          |
| 10     | ~87%      | ~41%    | low                      | high                              |

As epochs increase, training accuracy climbs steadily but validation accuracy lags far
behind — a classic **high-variance / overfitting** pattern. Ways to address this:

- Data augmentation (random crops, flips, brightness/contrast jitter)
- Stronger regularization (dropout, weight decay, label smoothing)
- Hyperparameter search over batch size, optimizer, learning rate, and initializer
- More training data per driver / better class balance
- Early stopping based on validation loss


## 8. Train Final Model & Predict on Holdout Set

In [ ]:
FINAL_EPOCHS = 10
FINAL_BATCH_SIZE = 32
MODEL_PATH = os.path.join(MODEL_DIR, f"resnet50_e{FINAL_EPOCHS}.h5")

final_model = ResNet50(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), classes=NUM_CLASSES)
final_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
final_model.fit(X_train, Y_train, epochs=FINAL_EPOCHS, batch_size=FINAL_BATCH_SIZE)

save_model(final_model, MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")


In [ ]:
# Reload to verify the save/load round-trip works correctly
final_model = load_model(MODEL_PATH)
print(f"{type(final_model).__name__} loaded successfully from {MODEL_PATH}")


In [ ]:
holdout_imgs_df = pd.read_csv(TEST_FILENAMES_CSV)
holdout_imgs_df.rename(columns={"imagename": "img"}, inplace=True)

X_HOLDOUT_PATH = os.path.join(ARRAY_DIR, f"X_test_{IMG_HEIGHT}_{IMG_WIDTH}.npy")
if not os.path.exists(X_HOLDOUT_PATH):
    create_img_array(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS, holdout_imgs_df, "test")

X_holdout = np.load(X_HOLDOUT_PATH)
print(f"Holdout set shape: {X_holdout.shape}")


In [ ]:
probabilities = final_model.predict(X_holdout, batch_size=32)

results_path = os.path.join(RESULTS_DIR, "test_predictions.csv")
np.savetxt(results_path, probabilities, delimiter=",")
print(f"Saved predictions to {results_path}")


## 9. Visual Sanity Check on Predictions

In [ ]:
X_holdout_scaled = rescale(X_holdout)

sample_index = min(50000, len(X_holdout) - 1)
print_image(X_holdout_scaled, index=sample_index, Y=probabilities)
predicted_class = probabilities[sample_index].argmax()
print(f"Predicted class: c{predicted_class} ({CLASS_NAMES['c' + str(predicted_class)]})")


## Next Steps / Improvements

- Use `tf.keras.preprocessing.image.ImageDataGenerator` (or `tf.data` + `keras.layers` augmentation)
  to apply random flips/rotations/zooms and reduce overfitting.
- Try transfer learning with a pretrained ResNet50/MobileNet (ImageNet weights) and fine-tune
  only the top layers — likely to outperform training from scratch on a small dataset.
- Add early stopping and learning-rate scheduling callbacks.
- Track experiments with TensorBoard or a tool like Weights & Biases.
- Evaluate with a full classification report (per-class precision/recall) in addition to accuracy.
